In [3]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [4]:
import sys

In [5]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers.table_aware_recursive import TableAwareRecursiveChunker
from financial_qa.embedders import GigaEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop
from financial_qa.agent.gigachat_agent_loop import GigaChatAgentLoop
from financial_qa.evaluation import evaluate_async, load_jsonl

In [6]:
dotenv.load_dotenv('.env')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
if not GIGACHAT_CREDENTIALS:
    raise ValueError('GIGACHAT_CREDENTIALS is required')

DATASET_FILE = 'dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = 50

RAG_DB = 'tarec_chunk_giga_embeddings_r'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 10
EMBED_MODEL = 'EmbeddingsGigaR'
GIGACHAT_SCOPE = 'GIGACHAT_API_PERS'

GEN_MODEL = "google/gemma-4-26b-a4b-it"
QUERY_CONCURRENCY = 1

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
JUDGE_PROCESSES = 100  # None => one process per question
USE_GIGACHAT_JUDGE = True

In [7]:
all_records = load_jsonl(DATASET_FILE)

In [8]:
len(all_records)

449

In [9]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
    else:
        print('huy')
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))

449
Loaded 50 records (split=None)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


In [10]:
# chunker = SlidingWindowChunker(chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
chunker = TableAwareRecursiveChunker(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
embedder = GigaEmbedder(
    credentials=GIGACHAT_CREDENTIALS,
    model=EMBED_MODEL,
    scope=GIGACHAT_SCOPE,
)
rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/parsed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
)

store_dir = Path('indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc()
else:
    print(f'Using existing index at {store_dir}')

# loop = GigaChatAgentLoop(
#     rag=rag,
#     model=GEN_MODEL,
#     credentials=GIGACHAT_CREDENTIALS,
#     scope=GIGACHAT_SCOPE,
# )

loop = OpenRouterAgentLoop(rag=rag, model=GEN_MODEL)

Using existing index at indexes/tarec_chunk_giga_embeddings_r


In [11]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')

Querying agent:   0%|          | 0/50 [00:00<?, ?question/s]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 13s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…
[GigaEmbedder] 429 on embeddings (attempt 2/6), sleeping 18s…
[GigaEmbedder] 429 on embeddings (attempt 3/6), sleeping 25s…
[GigaEmbedder] 429 on embeddings (attempt 4/6), sleeping 34s…
[GigaEmbedder] 429 on embeddings (attempt 5/6), sleeping 53s…


Querying agent:   2%|▏         | 1/50 [03:34<2:55:09, 214.47s/question, avg_s=214.46, errors=0, last_conf=1.05e+4]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…
[GigaEmbedder] 429 on embeddings (attempt 2/6), sleeping 16s…
[GigaEmbedder] 429 on embeddings (attempt 3/6), sleeping 25s…
[GigaEmbedder] 429 on embeddings (attempt 4/6), sleeping 36s…
[GigaEmbedder] 429 on embeddings (attempt 5/6), sleeping 52s…


Querying agent:   4%|▍         | 2/50 [06:27<2:32:17, 190.37s/question, avg_s=193.98, errors=0, last_conf=1.03e+4]

...

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…


Querying agent:  12%|█▏        | 6/50 [10:48<48:52, 66.64s/question, avg_s=108.16, errors=0, last_conf=1.03e+4]  

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…
[GigaEmbedder] 429 on embeddings (attempt 2/6), sleeping 16s…
[GigaEmbedder] 429 on embeddings (attempt 3/6), sleeping 25s…
[GigaEmbedder] 429 on embeddings (attempt 4/6), sleeping 36s…
[GigaEmbedder] 429 on embeddings (attempt 5/6), sleeping 52s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 13s…
[GigaEmbedder] 429 on embeddings (attempt 2/6), sleeping 18s…
[GigaEmbedder] 429 on embeddings (attempt 3/6), sleeping 25s…
[GigaEmbedder] 429 on embeddings (attempt 4/6), sleeping 35s…
[GigaEmbedder] 429 on embeddings (attempt 5/6), sleeping 53s…


Querying agent: 100%|██████████| 50/50 [26:29<00:00, 31.79s/question, avg_s=31.79, errors=0, last_conf=None]      

Done: 50 answers, 0 errors


In [12]:
query_errors

[]

In [13]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 50/50 [00:04<00:00, 10.56question/s, accuracy=8.00%, correct=4, errors=0] 


In [14]:
result['correct'] / result['total']

0.08

In [15]:
for res in result['results'][0:10]:
    print(res)
    print('-' * 75)

{'question_id': 'q_00d660efcf3e4607', 'question': 'Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?', 'gold_answer': '1,151 тыс. белорусских рублей', 'predicted_answer': 'На основании предоставленных документов, в отчете ЗАО «Альфа-Банк» за 2025 год содержится упоминание о том, что в следующей таблице должны быть представлены суммы по договорам с финансовыми активами и обязательствами по состоянию на 31 декабря 2025 года, которые должны быть переведены на альтернативные процентные базовые ставки (в связи с воздействием реформы IBOR).\n\nОднако сама таблица с конкретными числовыми значениями для этой категории в доступных фрагментах текста отсутствует. В предоставленных данных содержатся только другие таблицы (например, анализ прочих финансовых активов или анализ недисконтированных денежных потоков по финансовым обязательствам), но не та, которая содержит суммы, подлежащие 

In [16]:
import shutil
shutil.make_archive("logs", "zip", "logs")

'/Users/yarsem/Programming/ai360_bank/ai360-financial-qa/logs.zip'